In [1]:
import pandas as pd

In [10]:
experiments = ['baseline', 'lukasiewicz', 'min', 'l1', 'entropy', 'sign_ste', 'sign_sigmoid', 'temp_linear', 'temp_logarithmic']  # 'min'
domains = ['MIS', 'MaxCut']
dregs = [3, 5]
graph_sizes = [100]

In [16]:
# result_dict = {
#     key: {} for key in ['experiment', 'domain', 'd_regular', 'graph_size'] + ['qubo_loss', 'pred_size', 'solver_size', 'violation', 'improvement']
# }
result_df = pd.DataFrame(columns=['graph_id', 'rnd_seed', 'experiment', 'domain', 'd_regular', 'graph_size', 'qubo_loss', 'pred_size', 'solver_size', 'violation', 'improvement'])
for exp in experiments:
    for dom in domains:
        for dreg in dregs:
            for gsize in graph_sizes:
                for rnd_seed in range(10):
                    curr_df = pd.read_csv(f'perf_results/{exp}/{dom}/{dreg}/{gsize}/{rnd_seed}.csv')
                    curr_df = curr_df.assign(rnd_seed=rnd_seed, experiment=exp, domain=dom, d_regular=dreg, graph_size=gsize)
                    curr_df = curr_df.reset_index(names='graph_id')
                    result_df = pd.concat([result_df, curr_df])
print(result_df.head())

C:\Users\Martin\AppData\Local\Temp\ipykernel_11572\3580383226.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result_df = pd.concat([result_df, curr_df])


  graph_id rnd_seed experiment domain d_regular graph_size  qubo_loss  \
0        0        0   baseline    MIS         3        100 -41.974800   
1        1        0   baseline    MIS         3        100 -39.976002   
2        2        0   baseline    MIS         3        100 -42.974201   
3        3        0   baseline    MIS         3        100 -40.975399   
4        4        0   baseline    MIS         3        100 -39.975998   

  pred_size solver_size violation improvement  
0        42          38         0           4  
1        40          35         0           5  
2        43          37         0           6  
3        41          37         0           4  
4        40          40         0           0  


In [106]:
def get_best_of_seeds(df, metric, is_max=True):
    df = df[df['violation'] == 0]
    group = df.groupby(['experiment', 'graph_id'])[metric]
    if is_max:
        result = group.max()
    else:
        result = group.min()
    return result

def get_avg_of_seeds(df, metric):
    df = df[df['violation'] == 0]
    group = df.groupby(['experiment', 'graph_id'])[metric]
    mean, std = group.mean(), group.std()
    return mean, std

def find_best_per_graph(best_df, metric, is_max=True):
    best_records = []
    for i in range(20):
        filtered = best_df.xs(i, level=1)
        best_index = filtered.idxmax() if is_max else filtered.idxmin()
        best_record = filtered.loc[best_index]    
        best_records.append([i, best_index, best_record])
    best_per_graph_df = pd.DataFrame(best_records, columns=['graph_id', 'experiment', metric])
    return best_per_graph_df

def find_best_average(best_df, metric, is_max=True):
    mean = best_df.groupby('experiment')[metric].mean()
    best_index = mean.idxmax() if is_max else mean.idxmin()
    best_record = mean.loc[best_index]
    return best_index, best_record

## 1. MIS, d=3, gsize=100

In [101]:
df1 = result_df[(result_df['domain'] == 'MIS') & (result_df['d_regular'] == 3) & (result_df['graph_size'] == 100)]

In [102]:
best_df1 = get_best_of_seeds(df1, 'pred_size', is_max=True)
best_per_graph_df1 = find_best_per_graph(best_df1, 'pred_size', is_max=True)
print(best_per_graph_df1)

    graph_id experiment  pred_size
0          0   baseline         44
1          1        min         43
2          2   baseline         44
3          3   baseline         44
4          4   baseline         44
5          5         l1         44
6          6         l1         44
7          7   baseline         43
8          8   baseline         43
9          9   baseline         44
10        10   baseline         43
11        11   baseline         42
12        12   baseline         42
13        13   baseline         43
14        14   baseline         43
15        15   baseline         43
16        16   baseline         43
17        17   baseline         44
18        18   baseline         43
19        19   baseline         44


In [104]:
avg_df1, _ = get_avg_of_seeds(df1, 'pred_size')
best_per_graph_avg_df1 = find_best_per_graph(avg_df1, 'pred_size', is_max=True)
print(best_per_graph_avg_df1)

    graph_id experiment  pred_size
0          0   baseline  41.700000
1          1   baseline  40.900000
2          2   baseline  41.800000
3          3   baseline  42.000000
4          4   baseline  41.400000
5          5   baseline  41.900000
6          6   baseline  41.000000
7          7         l1  41.700000
8          8   baseline  41.300000
9          9   baseline  42.300000
10        10   baseline  41.666667
11        11   baseline  40.800000
12        12   baseline  41.111111
13        13   baseline  41.200000
14        14   baseline  41.500000
15        15   baseline  41.800000
16        16   baseline  41.900000
17        17   baseline  42.100000
18        18   baseline  41.700000
19        19   baseline  42.600000


In [107]:
best_idx1, best_rec1 = find_best_average(df1, 'pred_size', is_max=True)
print(best_idx1, best_rec1)

baseline 41.615


## 2. MaxCut, d=3, gsize=100

In [85]:
df2 = result_df[(result_df['domain'] == 'MaxCut') & (result_df['d_regular'] == 3) & (result_df['graph_size'] == 100)]

In [86]:
best_df2 = get_best_of_seeds(df2, 'pred_size', is_max=True)
best_per_graph_df2 = find_best_per_graph(best_df2, 'pred_size', is_max=True)
print(best_per_graph_df2)

    graph_id   experiment  pred_size
0          0     baseline        131
1          1     baseline        127
2          2     baseline        132
3          3     baseline        132
4          4           l1        135
5          5  temp_linear        131
6          6     baseline        129
7          7     baseline        129
8          8     baseline        130
9          9           l1        132
10        10     baseline        132
11        11     baseline        128
12        12     baseline        129
13        13     baseline        131
14        14     baseline        129
15        15     baseline        130
16        16     baseline        130
17        17           l1        132
18        18           l1        132
19        19     baseline        132


In [87]:
avg_df2, _ = get_avg_of_seeds(df2, 'pred_size')
best_per_graph_avg_df2 = find_best_per_graph(avg_df2, 'pred_size', is_max=True)
print(best_per_graph_avg_df2)

    graph_id experiment  pred_size
0          0   baseline      126.7
1          1   baseline      123.7
2          2         l1      125.1
3          3   baseline      127.5
4          4   baseline      123.7
5          5   baseline      125.5
6          6   baseline      123.4
7          7   baseline      124.5
8          8   baseline      124.0
9          9   baseline      127.0
10        10   baseline      126.0
11        11   baseline      123.8
12        12   baseline      124.6
13        13   baseline      124.3
14        14   baseline      125.2
15        15   baseline      125.6
16        16         l1      126.6
17        17         l1      126.1
18        18   baseline      126.9
19        19         l1      128.3


In [108]:
best_idx2, best_rec2 = find_best_average(df2, 'pred_size', is_max=True)
print(best_idx2, best_rec2)

baseline 125.34


## 3. MIS, d=5, gsize=100

In [88]:
df3 = result_df[(result_df['domain'] == 'MIS') & (result_df['d_regular'] == 5) & (result_df['graph_size'] == 100)]

In [89]:
best_df3 = get_best_of_seeds(df3, 'pred_size', is_max=True)
best_per_graph_df3 = find_best_per_graph(best_df3, 'pred_size', is_max=True)
print(best_per_graph_df3)

    graph_id   experiment  pred_size
0          0     baseline         36
1          1     baseline         38
2          2  lukasiewicz         37
3          3          min         36
4          4     baseline         36
5          5     baseline         36
6          6     baseline         37
7          7     baseline         37
8          8     baseline         37
9          9          min         35
10        10     baseline         35
11        11     baseline         37
12        12     baseline         36
13        13     baseline         36
14        14     baseline         37
15        15  lukasiewicz         36
16        16     baseline         36
17        17     baseline         37
18        18     baseline         37
19        19     baseline         35


In [90]:
avg_df3, _ = get_avg_of_seeds(df3, 'pred_size')
best_per_graph_avg_df3 = find_best_per_graph(avg_df3, 'pred_size', is_max=True)
print(best_per_graph_avg_df3)

    graph_id experiment  pred_size
0          0        min  33.600000
1          1   baseline  34.100000
2          2   baseline  34.800000
3          3   baseline  33.500000
4          4   baseline  33.900000
5          5        min  32.600000
6          6   baseline  35.300000
7          7   baseline  34.300000
8          8   baseline  35.200000
9          9        min  33.500000
10        10   baseline  32.600000
11        11        min  33.400000
12        12        min  33.500000
13        13   baseline  33.400000
14        14        min  33.700000
15        15   baseline  33.300000
16        16   baseline  34.600000
17        17   baseline  34.500000
18        18   baseline  34.666667
19        19   baseline  33.400000


In [109]:
best_idx3, best_rec3 = find_best_average(df3, 'pred_size', is_max=True)
print(best_idx3, best_rec3)

min 33.26


## 4.MaxCut, d=5, gsize=100

In [91]:
df4 = result_df[(result_df['domain'] == 'MaxCut') & (result_df['d_regular'] == 5) & (result_df['graph_size'] == 100)]

In [92]:
best_df4 = get_best_of_seeds(df4, 'pred_size', is_max=True)
best_per_graph_df4 = find_best_per_graph(best_df4, 'pred_size', is_max=True)
print(best_per_graph_df4)

    graph_id   experiment  pred_size
0          0          min        199
1          1          min        202
2          2          min        196
3          3          min        199
4          4          min        195
5          5          min        193
6          6          min        201
7          7          min        197
8          8          min        203
9          9          min        195
10        10          min        198
11        11     baseline        197
12        12          min        194
13        13          min        197
14        14          min        199
15        15          min        194
16        16          min        201
17        17  lukasiewicz        202
18        18          min        200
19        19           l1        195


In [93]:
avg_df4, _ = get_avg_of_seeds(df4, 'pred_size')
best_per_graph_avg_df4 = find_best_per_graph(avg_df4, 'pred_size', is_max=True)
print(best_per_graph_avg_df4)

    graph_id experiment  pred_size
0          0        min      192.9
1          1        min      192.6
2          2        min      190.1
3          3        min      191.6
4          4        min      190.1
5          5        min      188.0
6          6        min      193.4
7          7        min      192.2
8          8   baseline      193.2
9          9        min      190.1
10        10        min      191.2
11        11        min      188.7
12        12        min      191.4
13        13        min      190.6
14        14        min      191.7
15        15        min      188.7
16        16        min      192.4
17        17        min      191.2
18        18        min      192.6
19        19        min      188.4


In [110]:
best_idx4, best_rec4 = find_best_average(df4, 'pred_size', is_max=True)
print(best_idx4, best_rec4)

min 191.045
